# 05 · Transfer and data access (Block E)

An agreement says data *may* move. A **transfer process** makes it move.

## Concept: transfer processes, pull and push

A transfer is another asynchronous DSP exchange, tracked on each side by a **TransferProcess**:

```
consumer                                  provider
   │── TransferRequestMessage ───────────►│  agreement still valid? policy in scope transfer.process
   │                                       │  → asks its data plane to prepare the flow
   │◄──────────── TransferStartMessage ────│  (for PULL: carries the EDR)
   │       ... data flows ...              │
   │◄── TransferTermination / Completion ──│
```

The **transfer type** is `<data address type>-<PULL|PUSH>`:

- **PULL** (`HttpData-PULL`): the provider gives the consumer an access point, and the consumer fetches data whenever it wants while the transfer is `STARTED`;
- **PUSH** (`AzureStorage-PUSH`, `HttpData-PUSH`): the provider's data plane writes the data to a destination the consumer specified, then the transfer is `COMPLETED`.

## Concept: the EDR (Endpoint Data Reference)

For a pull transfer the provider sends an **EDR**: the URL of its data plane's public API plus a **short-lived token** signed by the provider.
The consumer's control plane stores it. Whoever holds the EDR can call the provider's data plane; the token tells the data plane which transfer, and therefore which data address, to serve.

This deployment adds a convenience on the consumer side, the DSE **data gateway** (`/consumer/dp/data`).
An application passes only the agreement id; the gateway looks up the newest EDR for it and forwards the request to the provider with the token.

In [1]:
import dataspace as ds

run_id = ds.recall("run_id", produced_by="00_environment.ipynb")
consumer_agreement_id = ds.recall("consumer_agreement_id", produced_by="04_negotiation.ipynb")
short_lived_asset = ds.recall("short_lived_asset", produced_by="02_publishing.ipynb")
consumer_api = ds.management_api("consumer")
provider_api = ds.management_api("provider")

## E1 · Start a pull transfer

**What:** `POST /consumer/cp/management/v3/transferprocesses`.

| Field | Meaning |
|---|---|
| `contractId` | the consumer's agreement id: which contract allows this transfer |
| `connectorId`, `counterPartyAddress`, `protocol` | the provider, and how to reach it |
| `transferType` | `HttpData-PULL`, one of the `distribution` formats the catalog listed |

**Returns:** `IdResponse` with the transfer process id. We then wait for `STARTED`, the state in which the EDR has arrived.

In [2]:
transfer_request = {
    "@context": ds.EDC_CONTEXT,
    "@type": "TransferRequest",
    "counterPartyAddress": ds.dsp_address("provider"),
    "protocol": ds.DSP_PROTOCOL,
    "connectorId": ds.did("provider"),
    "contractId": consumer_agreement_id,
    "transferType": "HttpData-PULL",
}
transfer_id = ds.post(f"{consumer_api}/transferprocesses", transfer_request, expect={200}).json()["@id"]


def transfer_state():
    return ds.get(f"{consumer_api}/transferprocesses/{transfer_id}/state", quiet=True).json()["state"]


ds.wait_for_state(transfer_state, {"STARTED", "TERMINATED"})

POST   /consumer/cp/management/v3/transferprocesses  ->  200


   states: INITIAL -> REQUESTING -> REQUESTED -> STARTED


'STARTED'

### The transfer process record

**What:** `GET /v3/transferprocesses/<id>`.  **Returns:** `state`, `type` (`CONSUMER`), `assetId`, `contractId`, `transferType`, and
`correlationId`, the provider's id for the same transfer.

In [3]:
transfer = ds.get(f"{consumer_api}/transferprocesses/{transfer_id}").json()
for field in ["state", "type", "assetId", "contractId", "transferType", "correlationId"]:
    print(f"   {field:15} {transfer.get(field)}")

GET    /consumer/cp/management/v3/transferprocesses/f5a76299-6c79-4bcf-b248-e840f3412a99  ->  200
   state           STARTED
   type            CONSUMER
   assetId         ging-data-1d0d4e
   contractId      599d9a9f-a70c-43e1-9a6a-e34d115cf6a1
   transferType    HttpData-PULL
   correlationId   8096b8f3-8ba1-4b64-90bd-00964c0fdc60


### The EDR the consumer received

**What:** `GET /v3/edrs/<transferProcessId>/dataaddress` on the consumer.  **Returns:** the EDR as a data address:

| Field | Meaning |
|---|---|
| `endpoint` | the provider's data plane public API, `http://provider-dataplane:8181/api/public/` (in-cluster) |
| `authType` | `bearer` |
| `authorization` | the access token: a JWT signed by the provider's key (`kid: provider-pub`), with the provider as issuer and the consumer as audience |

In [4]:
edr = ds.get(f"{consumer_api}/edrs/{transfer_id}/dataaddress").json()
print("endpoint:", edr["endpoint"])
print("authType:", edr["authType"])

token = edr["authorization"]
print("token header: ", ds.decode_jwt_header(token))
print("token claims: ", ds.decode_jwt_payload(token))

GET    /consumer/cp/management/v3/edrs/f5a76299-6c79-4bcf-b248-e840f3412a99/dataaddress  ->  200
endpoint: http://provider-dataplane:8181/api/public/
authType: bearer
token header:  {'kid': 'provider-pub', 'alg': 'ES256'}
token claims:  {'iss': 'did:web:provider-identityhub%3A8383:api:did', 'aud': 'did:web:consumer-identityhub%3A8383:api:did', 'sub': 'did:web:provider-identityhub%3A8383:api:did', 'iat': 1791554796, 'jti': 'eaea6a2e-caa9-409e-970a-865d7856e523'}


## E1 · Read the data through the consumer's gateway

**What:** `GET /consumer/dp/data?...` on the **consumer's** data plane, with two headers, both set to the consumer's agreement id:

| Header | Used for |
|---|---|
| `Authorization` | selecting the agreement, and therefore its newest EDR |
| `Contract-Id` | consumption records for billing (notebook 06). Without it the gateway answers 500 |

The gateway forwards path, query, method and body to the EDR endpoint with the token. The provider's data plane verifies the token and calls the asset's `baseUrl`.

**Returns:** whatever the backend returns. The sample backend echoes the `message` parameter.

In [5]:
response = ds.call(
    "GET",
    f"{ds.BASE_URL}/consumer/dp/data",
    params={"message": f"hello from run {run_id}"},
    headers={"Authorization": consumer_agreement_id, "Contract-Id": consumer_agreement_id},
)
print(response.text)

GET    /consumer/dp/data  ->  200
{"message":"hello from run 1d0d4e"}


## Concept: stopping access

Access ends when the transfer is terminated: the consumer loses its EDR and the provider's data plane stops accepting the token. Two ways appear here:

- **Retirement** (E6, a DSE extension): the provider withdraws an agreement on purpose. It is reversible.
- **Policy monitor** (E5): a control-plane loop re-evaluates running transfers in the `policy.monitor` scope and terminates them when the policy no longer holds, for example an expired `inForceDate`.

## E6 · The provider retires the agreement

Retirement is a **provider** operation, and it uses the **provider's** local id for the agreement.
The consumer transfer's `correlationId` is the provider's transfer id, and that provider transfer record names the provider's agreement in its `contractId`.

In [6]:
provider_transfer = ds.get(f"{provider_api}/transferprocesses/{transfer['correlationId']}").json()
provider_agreement_id = provider_transfer["contractId"]

print("provider's transfer:    ", provider_transfer["@id"], f"(type {provider_transfer['type']})")
print("consumer's agreement id:", consumer_agreement_id)
print("provider's agreement id:", provider_agreement_id)

GET    /provider/cp/management/v3/transferprocesses/8096b8f3-8ba1-4b64-90bd-00964c0fdc60  ->  200
provider's transfer:     8096b8f3-8ba1-4b64-90bd-00964c0fdc60 (type PROVIDER)
consumer's agreement id: 599d9a9f-a70c-43e1-9a6a-e34d115cf6a1
provider's agreement id: 2228df2c-0d34-4061-b07f-7d7f5b02173f


**What:** `POST /provider/cp/management/v3/contractagreements/retirements` with the agreement id and a reason.  **Returns:** 204.
The `@context` maps `edc:` and `dse:` because the body mixes the two vocabularies.

In [7]:
retirement = {
    "@context": {"dse": "https://w3id.org/dse/v0.0.1/ns/", "edc": ds.EDC},
    "edc:agreementId": provider_agreement_id,
    "dse:reason": f"tutorial run {run_id}",
}
ds.post(f"{provider_api}/contractagreements/retirements", retirement, expect={204})

POST   /provider/cp/management/v3/contractagreements/retirements  ->  204


<Response [204]>

The provider terminates the running transfer, and the consumer removes its EDR. The gateway now answers **403** because no EDR matches the agreement any more.

In [8]:
def access_is_blocked():
    status, _ = ds.read_data(consumer_agreement_id, message="anyone there?")
    return status == 403


ds.wait_until("data access blocked", access_is_blocked)
print("gateway says:  ", ds.read_data(consumer_agreement_id, message="anyone there?"))
print("transfer state:", transfer_state())

   data access blocked: yes, after 5s
gateway says:   (403, '{"errors":["No EDR satisfying criterion: agreementId = 599d9a9f-a70c-43e1-9a6a-e34d115cf6a1"]}')
transfer state: TERMINATED


**What:** `POST .../contractagreements/retirements/request` lists retired agreements, with their reason and date.

In [9]:
retirements = ds.post(f"{provider_api}/contractagreements/retirements/request", {"@context": ds.EDC_CONTEXT, "@type": "QuerySpec"}).json()
print(len(retirements), "retired agreement(s) on the provider")
ds.show(retirements[-1], max_chars=600)

POST   /provider/cp/management/v3/contractagreements/retirements/request  ->  200
1 retired agreement(s) on the provider
{
  "https://w3id.org/edc/v0.0.1/ns/agreementId": "2228df2c-0d34-4061-b07f-7d7f5b02173f",
  "https://w3id.org/dse/v0.0.1/ns/reason": "tutorial run 1d0d4e",
  "https://w3id.org/dse/v0.0.1/ns/agreementRetirementDate": 1791554797
}


## E6 · Reactivate it

**What:** `DELETE .../contractagreements/retirements/<provider agreement id>`.  **Returns:** 204. The agreement is valid again.

The old transfer stays `TERMINATED`, because terminated is a final state, so a **new** transfer is needed. `ds.start_pull_transfer()` repeats the E1 request.

In [10]:
ds.delete(f"{provider_api}/contractagreements/retirements/{provider_agreement_id}", expect={204})

second_transfer_id = ds.start_pull_transfer(consumer_agreement_id)
print(ds.read_data(consumer_agreement_id, message="back again"))

DELETE /provider/cp/management/v3/contractagreements/retirements/2228df2c-0d34-4061-b07f-7d7f5b02173f  ->  204
POST   /consumer/cp/management/v3/transferprocesses  ->  200


   states: INITIAL -> REQUESTED -> STARTUP_REQUESTED -> STARTED
(200, '{"message":"back again"}')


## E5 · An agreement that expires after 20 seconds

The `20s` entry has `inForceDate lteq contractAgreement+20s`. We negotiate it, start a transfer and read data straight away.
Then we wait: once 20 seconds have passed since `contractSigningDate`, the policy monitor terminates the transfer.

In [11]:
_, final_state, short_lived_agreement_id = ds.negotiate(short_lived_asset)
short_lived_transfer_id = ds.start_pull_transfer(short_lived_agreement_id)
print("right after the agreement:", ds.read_data(short_lived_agreement_id, message="still valid"))

POST   /consumer/cp/management/v3/contractnegotiations  ->  200


   states: INITIAL -> REQUESTED -> AGREED -> VERIFIED -> FINALIZED
POST   /consumer/cp/management/v3/transferprocesses  ->  200


   states: INITIAL -> REQUESTING -> REQUESTED -> STARTUP_REQUESTED -> STARTED


right after the agreement: (200, '{"message":"still valid"}')


In [12]:
def short_lived_transfer_state():
    return ds.get(f"{consumer_api}/transferprocesses/{short_lived_transfer_id}/state", quiet=True).json()["state"]


def short_lived_access_is_blocked():
    status, _ = ds.read_data(short_lived_agreement_id)
    return status == 403


ds.wait_for_state(short_lived_transfer_state, {"TERMINATED"}, timeout=120, interval=2)
ds.wait_until("data access blocked", short_lived_access_is_blocked)
print("after expiry:", ds.read_data(short_lived_agreement_id, message="too late"))

   states: STARTED -> TERMINATING_REQUESTED -> TERMINATED
   data access blocked: yes, after 0s
after expiry: (403, '{"errors":["No EDR satisfying criterion: agreementId = f0688931-9f38-48c0-af1a-118ef06b73b5"]}')


In [13]:
ds.remember(provider_agreement_id=provider_agreement_id, transfer_ids=[transfer_id, second_transfer_id, short_lived_transfer_id])

   remembered provider_agreement_id = 2228df2c-0d34-4061-b07f-7d7f5b02173f
   remembered transfer_ids = ['f5a76299-6c79-4bcf-b248-e840f3412a99', '73356fdd-7f15-42ec-a799-84f33551a93e', '61bcbd2c-457a-4166-bc80-b51c4ba52950']


Next: **06_telemetry_billing.ipynb**